# 03 · Class Creation, `__init_subclass__`, `__new__` & Metaclasses

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 60 min

> *"Metaclasses are deeper magic than 99% of users should ever worry about."* (Tim Peters). This lesson shows how they work **and** the simpler tools that usually replace them.

> 💡 New to the idea that classes are objects made by `type`? Start with the gentler introduction in [Module 06 · 08](../06-Object-Oriented-Programming/08-Everything-Is-an-Object-Classes-Types-and-Metaclasses.ipynb).

### 🎯 What you will learn
- Classes are objects created **at runtime** by `type`
- Creating classes dynamically with `type(name, bases, namespace)`
- The full class-creation sequence: `__prepare__` → body → metaclass `__new__`/`__init__` → `__set_name__` → `__init_subclass__`
- `__new__` vs `__init__` for instances (immutables, singletons, caching)
- ✅ `__init_subclass__` and class decorators: the simple alternatives
- Writing metaclasses: registries, validation, `__call__` control, ordered namespaces
- Real-world examples: `ABCMeta`, `Enum`, ORMs, Pydantic; when *not* to use metaclasses

## 1. A `class` statement is executed at runtime

In [1]:
print("before the class")
class Greeter:
    print("  executing the class body!")
    greeting = "hello"
    def greet(self, name):
        return f"{self.greeting}, {name}"
print("after the class:", Greeter, type(Greeter))

before the class
  executing the class body!
after the class: <class '__main__.Greeter'> <class 'type'>


## 2. Creating classes with `type(name, bases, namespace)`

`type` with one argument tells you an object's type; with **three** it **creates a class**:

In [2]:
def greet(self, name):
    return f"{self.greeting}, {name}"

DynamicGreeter = type("DynamicGreeter", (object,), {"greeting": "namaste", "greet": greet})
print(DynamicGreeter().greet("Asha"), DynamicGreeter.__name__)

# Useful for generating classes from data, e.g. one exception class per error code:
errors = {"NotFound": 404, "Forbidden": 403}
generated = {name: type(name, (Exception,), {"status": code}) for name, code in errors.items()}
try:
    raise generated["NotFound"]("missing page")
except Exception as e:
    print(type(e).__name__, e.status, e)

namaste, Asha DynamicGreeter
NotFound 404 missing page


## 3. `__new__` vs `__init__` for instances

- `__new__(cls, ...)` **creates** and returns the instance (a static method by default)
- `__init__(self, ...)` **initialises** it (returns `None`)
- If `__new__` returns an object that isn't an instance of `cls`, `__init__` is skipped

Use `__new__` for: subclassing **immutable** types, singletons, instance caching/interning.

In [3]:
class Celsius(float):                       # float is immutable → the value must be set in __new__
    def __new__(cls, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        return super().__new__(cls, value)
    def to_f(self):
        return self * 9 / 5 + 32

t = Celsius(25)
print(t, t + 1, t.to_f(), isinstance(t, float))

class Color:
    _cache = {}
    def __new__(cls, name):                  # instance caching: one object per name
        if name not in cls._cache:
            print(f"  creating {name}")
            cls._cache[name] = super().__new__(cls)
        return cls._cache[name]
    def __init__(self, name):
        self.name = name

print(Color("red") is Color("red"), Color("blue") is Color("red"))

25.0 26.0 77.0 True
  creating red
  creating blue
True False


## 4. The class-creation sequence

When Python runs `class Child(Base, metaclass=Meta, flag=True): ...`:

1. Determine the **metaclass** (explicit, or the most derived metaclass of the bases; default `type`)
2. `namespace = Meta.__prepare__(name, bases, **kwargs)` → the dict the body will fill
3. Execute the **class body** in that namespace
4. `cls = Meta.__new__(Meta, name, bases, namespace, **kwargs)`, which inside `type.__new__`:
   - calls `__set_name__(cls, attr_name)` on every attribute that defines it (descriptors, lesson 02)
   - calls `Base.__init_subclass__(**kwargs)` on the **parent** class
5. `Meta.__init__(cls, name, bases, namespace, **kwargs)`
6. Apply **class decorators** (bottom-up)
7. Bind the name `Child` to the result

In [4]:
class Meta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kw):
        print(f"1. __prepare__  {name}")
        return {}
    def __new__(mcls, name, bases, ns, **kw):
        print(f"3. Meta.__new__ {name}  attrs={[k for k in ns if not k.startswith('__')]}")
        return super().__new__(mcls, name, bases, ns, **kw)   # type.__new__ forwards kw to __init_subclass__
    def __init__(cls, name, bases, ns, **kw):
        print(f"5. Meta.__init__ {name}")
        super().__init__(name, bases, ns)

class Named:
    def __set_name__(self, owner, name):
        print(f"4a. __set_name__ {owner.__name__}.{name}")

class Base(metaclass=Meta):
    def __init_subclass__(cls, **kw):
        print(f"4b. Base.__init_subclass__({cls.__name__}, {kw})")
        super().__init_subclass__()

def decorate(cls):
    print(f"6. class decorator on {cls.__name__}")
    return cls

print("---- defining Child ----")
@decorate
class Child(Base, flag=True):
    print("2. executing the class body")
    field = Named()

1. __prepare__  Base
3. Meta.__new__ Base  attrs=[]
5. Meta.__init__ Base
---- defining Child ----
1. __prepare__  Child
2. executing the class body
3. Meta.__new__ Child  attrs=['field']
4a. __set_name__ Child.field
4b. Base.__init_subclass__(Child, {'flag': True})
5. Meta.__init__ Child
6. class decorator on Child


## 5. ✅ `__init_subclass__`: hooks for subclasses (no metaclass needed)

Runs every time a subclass is created, which is perfect for **plugin registries** and **enforcing rules**:

In [5]:
class Exporter:
    registry: dict[str, type] = {}

    def __init_subclass__(cls, /, fmt: str, **kwargs):      # keyword args from the class statement
        super().__init_subclass__(**kwargs)
        if not callable(getattr(cls, "export", None)):
            raise TypeError(f"{cls.__name__} must define export()")
        Exporter.registry[fmt] = cls

class CSVExporter(Exporter, fmt="csv"):
    def export(self, rows):
        return "\n".join(",".join(map(str, r)) for r in rows)

class JSONExporter(Exporter, fmt="json"):
    def export(self, rows):
        import json
        return json.dumps(rows)

print(Exporter.registry)
rows = [[1, "a"], [2, "b"]]
for fmt in ("csv", "json"):
    print(fmt, "→", Exporter.registry[fmt]().export(rows))

try:
    class Broken(Exporter, fmt="xml"):
        pass
except TypeError as e:
    print("TypeError:", e)

{'csv': <class '__main__.CSVExporter'>, 'json': <class '__main__.JSONExporter'>}
csv → 1,a
2,b
json → [[1, "a"], [2, "b"]]
TypeError: Broken must define export()


## 6. ✅ Class decorators: modify a class after creation

In [6]:
def auto_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in vars(self).items() if not k.startswith("_"))
        return f"{cls.__name__}({fields})"
    cls.__repr__ = __repr__
    return cls

def frozen(cls):
    def __setattr__(self, name, value):
        if getattr(self, "_initialised", False):
            raise AttributeError(f"{cls.__name__} is frozen")
        object.__setattr__(self, name, value)
    original_init = cls.__init__
    def __init__(self, *args, **kwargs):
        original_init(self, *args, **kwargs)
        object.__setattr__(self, "_initialised", True)
    cls.__setattr__, cls.__init__ = __setattr__, __init__
    return cls

@auto_repr
@frozen
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

p = Point(1, 2)
print(p)
try:
    p.x = 10
except AttributeError as e:
    print("AttributeError:", e)

Point(x=1, y=2)
AttributeError: Point is frozen


`@dataclass`, `@functools.total_ordering` and `@typing.final` are class decorators too.

## 7. Writing metaclasses

A metaclass is a subclass of `type`. It controls **class creation** (`__new__`/`__init__`) and even **instance creation** (`__call__`).

### 7.1 Validating class definitions

In [7]:
class InterfaceMeta(type):
    required = ("run", "stop")
    def __new__(mcls, name, bases, ns):
        cls = super().__new__(mcls, name, bases, ns)
        if bases:                                          # skip the base class itself
            missing = [m for m in mcls.required if not callable(getattr(cls, m, None))]
            if missing:
                raise TypeError(f"{name} is missing {missing}")
        return cls

class Service(metaclass=InterfaceMeta):
    pass

class WebServer(Service):
    def run(self): return "running"
    def stop(self): return "stopped"

try:
    class Half(Service):
        def run(self): ...
except TypeError as e:
    print("TypeError:", e)

TypeError: Half is missing ['stop']


### 7.2 Controlling instantiation with `__call__`: a singleton

`Cls(...)` actually calls `type(Cls).__call__(Cls, ...)`, which normally runs `__new__` then `__init__`:

In [8]:
class SingletonMeta(type):
    _instances = {}
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Settings(metaclass=SingletonMeta):
    def __init__(self):
        print("  loading settings once")
        self.debug = True

a, b = Settings(), Settings()
print(a is b)

  loading settings once
True


> 💡 In Python, a **module** is already a singleton: often the simplest solution is a module-level object (`settings = Settings()`) rather than a metaclass.

### 7.3 A mini ORM: descriptors + metaclass together

In [9]:
class Column:
    def __init__(self, sql_type):
        self.sql_type = sql_type
    def __set_name__(self, owner, name):
        self.name = name

class ModelMeta(type):
    def __new__(mcls, name, bases, ns):
        cls = super().__new__(mcls, name, bases, ns)
        cls._columns = {k: v for k, v in ns.items() if isinstance(v, Column)}
        cls._table = ns.get("__tablename__", name.lower() + "s")
        return cls

    def create_table_sql(cls):                 # methods on the metaclass become CLASS methods
        cols = ", ".join(f"{n} {c.sql_type}" for n, c in cls._columns.items())
        return f"CREATE TABLE {cls._table} ({cols});"

class Model(metaclass=ModelMeta):
    pass

class User(Model):
    id = Column("INTEGER PRIMARY KEY")
    name = Column("TEXT NOT NULL")
    email = Column("TEXT UNIQUE")

print(User.create_table_sql())
print(hasattr(User(), "create_table_sql"))    # not visible on instances

CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT UNIQUE);
False


### 7.4 Metaclass conflicts

A class can have only one metaclass; when bases use different metaclasses, Python needs one that inherits from both:

In [10]:
from abc import ABCMeta, abstractmethod

class Plugin(metaclass=ABCMeta):             # ABC uses the ABCMeta metaclass
    @abstractmethod
    def run(self): ...

try:
    class Weird(Plugin, Model):               # ABCMeta vs ModelMeta
        pass
except TypeError as e:
    print("TypeError:", str(e)[:90], "...")

class PluginModelMeta(ABCMeta, ModelMeta):   # combine them
    pass

class Fine(Plugin, Model, metaclass=PluginModelMeta):
    def run(self): return "ok"
print(Fine().run())

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of th ...
ok


## 8. Metaclasses in the wild
- **`abc.ABCMeta`**: tracks abstract methods, blocks instantiation (Module 06)
- **`enum.EnumType`**: turns class attributes into unique members, makes the class iterable
- **Django models / SQLAlchemy declarative**: collect fields into table metadata
- **Pydantic `BaseModel`**: builds validators and schemas from annotations (Module 15)

In [11]:
from enum import Enum
class Status(Enum):
    ACTIVE = 1
    INACTIVE = 2
print(type(Status), list(Status), Status(1))

<class 'enum.EnumType'> [<Status.ACTIVE: 1>, <Status.INACTIVE: 2>] Status.ACTIVE


## 9. Choosing the right tool

| Goal | Simplest tool |
|---|---|
| Add/modify methods on one class | **class decorator** |
| React to every subclass (registry, validation) | **`__init_subclass__`** |
| Reusable attribute behaviour | **descriptor** (`__set_name__`) |
| Customise instance creation | `__new__` / factory classmethod |
| Change *how classes behave* (operators on the class, `__call__`, namespace) | **metaclass** |

## 📝 Summary
- Classes are runtime objects created by `type`; `type(name, bases, ns)` makes classes dynamically.
- Creation sequence: `__prepare__` → body → `Meta.__new__` (→ `__set_name__`, `__init_subclass__`) → `Meta.__init__` → decorators.
- Prefer `__init_subclass__`, class decorators and descriptors; reach for metaclasses only when you must.
- Metaclass `__call__` controls instantiation; metaclass methods act as class-level methods.

## 🏋️ Practice
1. Use `__init_subclass__` to build a command registry: `class Hello(Command, name="hello")` → `Command.run("hello")`.
2. Write a class decorator `@count_instances` that adds a class attribute counting instances created.
3. Write a metaclass that makes every method name lowercase-only (raise `TypeError` for `CamelCase` method names).

<details><summary>Show solutions</summary>

```python
class Command:
    commands = {}
    def __init_subclass__(cls, name, **kw):
        super().__init_subclass__(**kw)
        Command.commands[name] = cls
    @classmethod
    def run(cls, name):
        return cls.commands[name]().execute()

class Hello(Command, name="hello"):
    def execute(self): return "Hello!"
print(Command.run("hello"))

def count_instances(cls):
    cls.count = 0
    init = cls.__init__
    def __init__(self, *a, **k):
        type(self).count += 1
        init(self, *a, **k)
    cls.__init__ = __init__
    return cls

class SnakeCaseMeta(type):
    def __new__(mcls, name, bases, ns):
        bad = [k for k, v in ns.items() if callable(v) and not k.startswith("__") and k != k.lower()]
        if bad:
            raise TypeError(f"method names must be lowercase: {bad}")
        return super().__new__(mcls, name, bases, ns)
```
</details>